In [ ]:
# infer.py
import argparse, os
import numpy as np
import cv2
import torch
import torch.nn.functional as F

from model import UNet11_4ch  # 你提供的 model.py

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def read_tif_as_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        # 若是 RGBA，先丟棄 A
        img = img[:, :, :3]
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def read_depth_optional(path, target_shape):
    if path is None:
        return None
    d = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if d is None:
        raise FileNotFoundError(f"Cannot read depth: {path}")
    if d.ndim == 3:
        d = cv2.cvtColor(d, cv2.COLOR_BGR2GRAY)
    d = cv2.resize(d, (target_shape[1], target_shape[0]), interpolation=cv2.INTER_NEAREST)
    return d

def make_4ch(rgb_u8, depth_u8=None):
    rgb = rgb_u8.astype(np.float32) / 255.0
    # RGB 正規化到 ImageNet
    rgb_norm = (rgb - IMAGENET_MEAN) / IMAGENET_STD
    if depth_u8 is None:
        # 用 RGB 的灰度均值當第4通道
        gray = cv2.cvtColor((rgb_u8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    else:
        # 將深度做 min-max 到 [0,1]
        d = depth_u8.astype(np.float32)
        mn, mx = float(np.min(d)), float(np.max(d))
        gray = (d - mn) / (mx - mn + 1e-8)
    # 第4通道用 mean=0.5, std=0.5 標準化到約 [-1,1]
    d_norm = (gray - 0.5) / 0.5
    x4 = np.dstack([rgb_norm, d_norm[..., None]])  # HWC, 4
    return x4

@torch.no_grad()
def run_infer(img_path, weights, depth_path=None, out_mask="mask.png", out_overlay="overlay.png",
              input_size=512, thresh=0.5, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # 讀圖
    rgb = read_tif_as_rgb(img_path)
    H, W = rgb.shape[:2]

    # 讀深度(可選)
    depth = read_depth_optional(depth_path, (H, W))

    # 建立 4ch 並 resize 到模型尺寸
    x4 = make_4ch(rgb, depth)
    x4_resized = cv2.resize(x4, (input_size, input_size), interpolation=cv2.INTER_LINEAR)

    # NHWC->NCHW 張量
    x = torch.from_numpy(x4_resized.transpose(2, 0, 1)).unsqueeze(0).float().to(device)

    # 模型
    model = UNet11_4ch(pretrained=False)  # 推論不載預訓練
    ckpt = torch.load(weights, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    model.to(device)
    model.eval()

    # 前向
    logits = model(x)             # [1,1,h,w]
    prob = torch.sigmoid(logits)  # [1,1,h,w]
    prob_up = F.interpolate(prob, size=(H, W), mode="bilinear", align_corners=False)
    prob_np = prob_up.squeeze().cpu().numpy().astype(np.float32)

    # 二值化
    mask = (prob_np >= float(thresh)).astype(np.uint8) * 255

    # 輸出遮罩
    cv2.imwrite(out_mask, mask)

    # 疊圖可視化
    overlay = (rgb.copy()).astype(np.uint8)
    color = np.zeros((H, W, 3), dtype=np.uint8)
    color[:, :, 0] = 0     # R
    color[:, :, 1] = 255   # G
    color[:, :, 2] = 0     # B
    alpha = 0.35
    idx = mask > 0
    overlay[idx] = cv2.addWeighted(overlay[idx], 1 - alpha, color[idx], alpha, 0)
    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.imwrite(out_overlay, overlay_bgr)

    print(f"Saved mask to {out_mask}")
    print(f"Saved overlay to {out_overlay}")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True, help=".tif 影像路徑")
    ap.add_argument("--weights", required=True, help="模型權重 .pth/.pt")
    ap.add_argument("--depth", default=None, help="可選：深度圖 .tif/.png")
    ap.add_argument("--out_mask", default="mask.png")
    ap.add_argument("--out_overlay", default="overlay.png")
    ap.add_argument("--size", type=int, default=512, help="模型輸入邊長")
    ap.add_argument("--th", type=float, default=0.5, help="二值化門檻")
    ap.add_argument("--device", default=None, help="cuda 或 cpu")
    args = ap.parse_args()

    run_infer(
        img_path=args.image,
        weights=args.weights,
        depth_path=args.depth,
        out_mask=args.out_mask,
        out_overlay=args.out_overlay,
        input_size=args.size,
        thresh=args.th,
        device=args.device,
    )

if __name__ == "__main__":
    main()


发生错误: Error(s) in loading state_dict for UNet11_4ch:
	Missing key(s) in state_dict: "encoder.0.bias", "conv1.bias", "center.block.0.block.0.bias", "center.block.1.bias", "dec5.block.0.block.0.bias", "dec5.block.1.bias", "dec4.block.0.block.0.bias", "dec4.block.1.bias", "dec3.block.0.block.0.bias", "dec3.block.1.bias", "dec2.block.0.block.0.bias", "dec2.block.1.bias", "dec1.block.0.bias". 
	Unexpected key(s) in state_dict: "inv.block.0.weight", "inv.block.1.weight", "inv.block.1.bias", "inv.block.1.running_mean", "inv.block.1.running_var", "inv.block.1.num_batches_tracked", "inv.block.3.weight", "inv.block.4.weight", "inv.block.4.bias", "inv.block.4.running_mean", "inv.block.4.running_var", "inv.block.4.num_batches_tracked", "inv.block.6.weight", "inv.block.7.weight", "inv.block.7.bias", "inv.block.7.running_mean", "inv.block.7.running_var", "inv.block.7.num_batches_tracked". 

请检查:
1. 模型文件路径是否正确
2. 测试图片路径是否正确
3. 深度图路径是否正确（或程序能否自动找到）
4. model.py文件是否在当前目录
5. 所需依赖是否已安装: torch, torchvision

C:\Users\op237\AppData\Local\Temp\ipykernel_26880\253016204.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)
